In [ ]:
import sys
import numpy as np
import importlib
import dragons_chime_conv_combine
import matplotlib.pyplot as plt
from astropy.io import fits
from astropy.wcs import WCS
from astropy.coordinates import SkyCoord, ICRS, Galactic
import astropy.units as u
from reproject import reproject_interp

In [ ]:
fpick = 651.5

#hdu = fits.open('/srv/data/gmims/gmims-lbn/experiments/data/chime_regrid_U.fits')
hdu = fits.open('/srv/aordog/CHIME_data/CHIME_FD_Oct_12_2021/U_CHIME_cube_new_high.fits')
#hdu = fits.open('/srv/data/gmims/gmims-lbn/experiments/data/dragons_regrid_U.fits')
hdr = hdu[0].header
wcs = WCS(hdr)
data = hdu[0].data

freq = wcs.all_pix2world(0,0,range(wcs.array_shape[0]),0)[2]/1e6
fidx = abs(fpick-freq).argmin()

print(data.shape)
print(freq[fidx])

data[np.isnan(data)] = 0.0

plt.imshow(data[fidx],vmin=-1,vmax=1)

In [ ]:
data_fft = np.fft.fft2(data[fidx])
data_fft_shift = np.fft.fftshift(data_fft)


fig,ax = plt.subplots(1,figsize=(30,15))

ax.imshow(abs(data_fft_shift),vmin=0,vmax=1000)


In [ ]:
fig,ax = plt.subplots(1,figsize=(30,5))
ax.plot(abs(data_fft_shift[102]))
#ax.set_xlim(500,2200)

In [ ]:
hdr2 = hdr.copy()
hdr2['CDELT1'] = hdr['CDELT1']/4
hdr2['CDELT2'] = hdr['CDELT2']/4
hdr2['NAXIS1'] = hdr['NAXIS1']*4
hdr2['NAXIS2'] = hdr['NAXIS2']*4
hdr2['CRPIX1'] = hdr['CRPIX1']*4
hdr2['CRPIX2'] = hdr['CRPIX2']*4

print(repr(hdr2))

map_regrid, footprint = reproject_interp((data[fidx,:,:],wcs.dropaxis(2)),WCS(hdr2).dropaxis(2))


In [ ]:
map_regrid[np.isnan(map_regrid)] = 0.0

In [ ]:
plt.imshow(map_regrid[3200:5760],vmin=-1,vmax=1)

In [ ]:
data_fft = np.fft.fft2(map_regrid[3200:5760])
data_fft_shift = np.fft.fftshift(data_fft)

fig,ax = plt.subplots(1,figsize=(30,15))

ax.imshow(abs(data_fft_shift),vmin=0,vmax=1000)

In [ ]:
importlib.reload(dragons_chime_conv_combine)

dxy = hdr2['CDELT2']
print(dxy)

map_FFT, mask, beam, xuv, yuv, map_deconv = dragons_chime_conv_combine.SA_deconvolve(map_regrid, 5.0, dxy, freq[fidx])

In [ ]:
fig,ax = plt.subplots(2,1,figsize=(20,16))

ax[0].imshow(np.log(abs(map_FFT)),vmin=0,vmax=10,cmap='cubehelix')
ax[1].imshow(beam,cmap='cubehelix')

In [ ]:
yidx = 2881
xidx = 2880*2+1
print(mask.shape)

fig,ax = plt.subplots(2,1,figsize=(16,12))
s = 5

ax[0].plot(xuv[yidx],beam[yidx])
ax2 = ax[0].twinx()
ax2.scatter(xuv[yidx],abs(map_FFT[yidx]),color='C1',s=s)
#ax2.scatter(xuv[yidx],mask[yidx]*abs(dragons_FFT[yidx])/beam[yidx],color='C2',s=s)
#ax2.set_ylim(1e1,1e5)
#ax2.set_yscale('log')

ax[1].plot(yuv[:,xidx],beam[:,xidx])
ax2 = ax[1].twinx()
ax2.scatter(yuv[:,xidx],abs(map_FFT[:,xidx]),color='C1',s=s)
#ax2.scatter(yuv[:,xidx],mask[:,xidx]*abs(dragons_FFT[:,xidx])/beam[:,xidx],color='C2',s=s)
ax2.set_ylim(1e1,1e5)
#ax2.set_yscale('log')


for i in range(0,2):
    ax[i].set_ylim(0,1)
    ax[i].grid()
    ax[i].set_xlim(-15,15)

In [ ]:
def make_diff_map(wcs, map1, map2, stokes = 'pi',
                  vmax=20, diff_frac=0.5, RA_range = [24,0],dec_range = [-30,90]):

    crange = SkyCoord(RA_range, dec_range, frame=ICRS, unit=(u.hourangle, u.deg))
    
    fs = 14
    width = 20
    
    height = 2.5*width*(dec_range[1]-dec_range[0])/((RA_range[0]-RA_range[1])*360/24)
    print(height)
    
    fig = plt.figure(figsize=(width,height))
    ax1 = fig.add_subplot(311, projection=wcs.celestial)
    ax2 = fig.add_subplot(312, projection=wcs.celestial)
    ax3 = fig.add_subplot(313, projection=wcs.celestial)

    if stokes == 'i':
        vmin = 0
        cmap = 'cubehelix'
    if stokes == 'pi':
        vmin = 0
        cmap = 'viridis'
    if ((stokes == 'q') or (stokes == 'u')):
        vmin = -vmax
        cmap = 'RdBu_r'

    im1 = ax1.imshow(map1,origin='lower',cmap=cmap,vmin=vmin,vmax=vmax)
    im2 = ax2.imshow(map2,origin='lower',cmap=cmap,vmin=vmin,vmax=vmax)
    im3 = ax3.imshow(map2-map1,origin='lower',cmap='RdBu_r',vmin=-vmax*diff_frac,vmax=vmax*diff_frac)
 
    axs = [ax1,ax2,ax3]
    ims = [im1,im2,im3]
    
    for i in range(0,len(axs)):
        axs[i].set_ylim(wcs.world_to_pixel(crange)[1])
        axs[i].set_xlim(wcs.world_to_pixel(crange)[0])
        cbar = fig.colorbar(ims[i], ax=axs[i], shrink=1, pad=0.01)
        axs[i].set_xlabel(' ')
        axs[i].set_ylabel('dec',fontsize=fs)
    ax3.set_xlabel('RA',fontsize=fs)

    return

In [ ]:
make_diff_map(wcs.dropaxis(2), dragons[fidx], dragons_deconv, stokes = 'q',
                  vmax=1, diff_frac=0.1, RA_range = [8,0],dec_range = [20,90])

In [ ]:
import scipy

def gf(mu,fwhm,x):
    return np.exp(-4*np.log(2)*((x-mu)**2)/fwhm**2)

In [ ]:
ptsy = np.array([8,8,7,7,8,9,8,7,6,5,4,5,6,5,3,2,2,3,4,5,6,5,4,2,2,3,3,7,15,14,14,15,15,16,15,15,14,15,16,15,15])
ptsx = np.arange(0,len(ptsy),1)

x = np.arange(0, len(ptsy)-1, 0.2)

f = scipy.interpolate.interp1d(ptsx,ptsy,kind='quadratic')
y = f(x) - np.nanmean(f(x))
print(x.shape)

beam = gf(0,20,x)

plt.plot(x,y)
plt.plot(x,beam*10)

In [ ]:
y_fft = np.fft.fft(y)
y_fft_shift = np.fft.fftshift(y_fft)

u = np.arange(0,len(x),1)

plt.scatter(u,abs(y_fft_shift),s=5)
plt.plot(u,abs(y_fft_shift))
plt.xlim(50,150)
print(y_fft.shape)

#lp = gf(100,5,u)
lp = np.fft.fftshift(np.fft.fft(beam))
lp = lp/np.nanmax(lp)

plt.plot(u,lp)
#print(lp.real)

y_fft_lp = y_fft_shift*lp
plt.plot(u,abs(y_fft_lp))

In [ ]:
y_filt = np.fft.ifft(np.fft.ifftshift(y_fft_lp))

plt.plot(x,y)
plt.plot(x,y_filt.real)

In [ ]:
y_filt_fft = np.fft.fft(y_filt.real)
y_filt_fft_shift = np.fft.fftshift(y_filt_fft)

u = np.arange(0,len(x),1)

plt.scatter(u,abs(y_filt_fft_shift),s=5)
plt.plot(u,abs(y_filt_fft_shift))
plt.xlim(50,150)

plt.plot(u,lp)
#print(lp.real)

y_filt_fft_fix = y_filt_fft_shift/lp
plt.plot(u,abs(y_filt_fft_fix))

In [ ]:
y_filt_fix = np.fft.ifft(np.fft.ifftshift(y_filt_fft_fix))

plt.plot(x,y)
plt.plot(x,y_filt.real)
plt.plot(x,y_filt_fix.real)

In [ ]:
hdu1 = fits.open('/srv/data/gmims/gmims-lbn/experiments/data/dragons_regrid_U.fits')
hdu2 = fits.open('/srv/data/dva/dragons_hoard/no_BLML_500-1000MHz/Mar05_2024_BB_600_to_700_fixed/merged_maps/high/u/After_Ground_Iono_690MHz_high_55.u.fits')

data1 = hdu1[0].data
data2 = hdu2[0].data

print(data1.shape)
print(data2.shape)

hdr1 = hdu1[0].header
hdr2 = hdu2[0].header

wcs = WCS(hdr1)
print(wcs)


In [ ]:
fpick = 690.5

freq = wcs.all_pix2world(0,0,range(wcs.array_shape[0]),0)[2]/1e6
fidx = abs(fpick-freq).argmin()
print(freq[fidx])

map1 = data1[fidx]
map2_Jy, footprint = reproject_interp((data2,WCS(hdr2)),wcs.dropaxis(2))

map1[np.isnan(map1)] = 0.0
map1[0:800] = 0.0
map2_Jy[np.isnan(map2_Jy)] = 0.0
map2_Jy[0:800] = 0.0

print(map1.shape)
print(map2_Jy.shape)

In [ ]:
JytoK = np.load('/srv/data/dva/conversion_factor_JytoK_July2024_BB.npy')
idx = abs(JytoK[0]-fpick).argmin()
print(JytoK[0][idx],JytoK[1][idx])

map2 = map2_Jy*JytoK[1][idx]


In [ ]:
fig,ax = plt.subplots(3,1,figsize=(20,25))

ax[0].imshow(map1,vmin=-1.5,vmax=1.5,cmap='RdBu_r',origin='lower')
ax[1].imshow(map2,vmin=-1.5,vmax=1.5,cmap='RdBu_r',origin='lower')
ax[2].imshow(map2-map1,vmin=-1,vmax=1,cmap='RdBu_r',origin='lower')

In [ ]:
map1_fft = np.fft.fft2(map1)
map2_fft = np.fft.fft2(map2)

map1_fft_shift = np.fft.fftshift(map1_fft)
map2_fft_shift = np.fft.fftshift(map2_fft)

fig,ax = plt.subplots(3,1,figsize=(20,45))

ax[0].imshow(abs(map1_fft_shift)[:,1440-400:1440+400],cmap='cubehelix',vmin=0,vmax=1000)
ax[1].imshow(abs(map2_fft_shift)[:,1440-400:1440+400],cmap='cubehelix',vmin=0,vmax=1000)
ax[2].imshow(abs(map2_fft_shift)[:,1440-400:1440+400]-abs(map1_fft_shift)[:,1440-400:1440+400],cmap='viridis',vmin=-1000,vmax=1000)

print(map1_fft_shift.shape)

In [ ]:
fig,ax = plt.subplots(1,1,figsize=(20,8))

ax.plot(abs(map1_fft_shift)[720])
ax.plot(abs(map2_fft_shift)[720])
ax.set_xlim(1440-600,1440+600)
ax.set_ylim(1,1e5)
ax.set_yscale('log')

In [ ]:
importlib.reload(dragons_chime_conv_combine)

dxy = hdr1['CDELT2']
print(dxy)

map_FFT1, mask1, beam1, xuv1, yuv1, map_deconv1 = dragons_chime_conv_combine.SA_deconvolve(map1[800:1440], 5.0, dxy, freq[fidx])
map_FFT2, mask2, beam2, xuv2, yuv2, map_deconv2 = dragons_chime_conv_combine.SA_deconvolve(map2[800:1440], 5.0, dxy, freq[fidx])

In [ ]:
print(map_FFT1.shape)
print(mask1.shape)
print(xuv1.shape)
print(beam1.shape)
print(freq[fidx])

In [ ]:
fig,ax = plt.subplots(3,1,figsize=(20,12))

ax[0].plot(xuv1[320],beam1[320])
ax[0].plot(xuv2[320],beam2[320])

ax[1].plot(xuv1[320],abs(map_FFT1[320]))
#ax[1].plot(xuv2[320],abs(map_FFT2[320]))
#ax[1].set_yscale('log')

ax[2].plot(xuv1[320],mask1[320]*abs(map_FFT1[320])/beam1[320])
ax[2].plot(xuv2[320],mask2[320]*abs(map_FFT2[320])/beam2[320])
#ax[2].set_yscale('log')

for i in range(0,3):
    ax[i].set_xlim(0,15)
    if i>0:
        ax[i].set_ylim(1,1e5)
        ax[i].set_ylim(0,500)

In [ ]:
fig,ax = plt.subplots(3,3,figsize=(20,12))

# not sharpened
ax[0,0].imshow(map1[800:1440,1800:2880],vmin=-1.5,vmax=1.5,cmap='RdBu_r',origin='lower')
ax[1,0].imshow(map2[800:1440,1800:2880],vmin=-1.5,vmax=1.5,cmap='RdBu_r',origin='lower')
ax[2,0].imshow(map2[800:1440,1800:2880]-map1[800:1440,1800:2880],vmin=-1,vmax=1,cmap='RdBu_r',origin='lower')

# sharpened
ax[0,1].imshow(map_deconv1[800:1440,1800:2880],vmin=-1.5,vmax=1.5,cmap='RdBu_r',origin='lower')
ax[1,1].imshow(map_deconv2[800:1440,1800:2880],vmin=-1.5,vmax=1.5,cmap='RdBu_r',origin='lower')
ax[2,1].imshow(map_deconv2[800:1440,1800:2880]-map_deconv1[800:1440,1800:2880],vmin=-1,vmax=1,cmap='RdBu_r',origin='lower')

ax[0,2].imshow(map_deconv1[800:1440,1800:2880]-map1[800:1440,1800:2880],vmin=-0.1,vmax=0.1,cmap='RdBu_r',origin='lower')
ax[1,2].imshow(map_deconv2[800:1440,1800:2880]-map2[800:1440,1800:2880],vmin=-0.1,vmax=0.1,cmap='RdBu_r',origin='lower')

In [ ]:
(1.22*(3e8/690e6)/15)*180/np.pi